# 09 — Batch prediction with saved models

Notebook này chạy suy luận từ dữ liệu CSV thô bằng hai model đã lưu: **credit risk** và **fraud risk score**. Không train hoặc thay đổi model.

Đầu vào cần có cột `customer_id` và các cột gốc giống schema dữ liệu train. Kết quả được ghi ra CSV trong `data/predictions/`.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs' / 'config.yaml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'configs' / 'config.yaml').is_file():
    raise FileNotFoundError('Không tìm thấy configs/config.yaml. Hãy mở notebook từ thư mục notebooks hoặc chỉnh PROJECT_ROOT.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipelines.inference_pipeline import run_batch_inference
from src.utils.config import load_config

## Chọn input và output

Đổi `INPUT_PATH` thành đường dẫn tới CSV của bạn. `TASK` có thể là `both`, `credit`, hoặc `fraud`.

In [2]:
config = load_config(PROJECT_ROOT / 'configs/config.yaml')

# Ví dụ nhỏ để kiểm tra pipeline. Thay bằng PROJECT_ROOT / 'data/raw/your_customers.csv' khi dự đoán dữ liệu của bạn.
INPUT_PATH = PROJECT_ROOT / 'tests/data/sample.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data/predictions/sample_predictions.csv'
TASK = 'both'  # 'both', 'credit', hoặc 'fraud'
MODEL_VERSION = config['inference']['model_version']  # v1.0.0

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy input: {INPUT_PATH}')

raw_preview = pd.read_csv(INPUT_PATH)
if 'customer_id' not in raw_preview.columns:
    raise ValueError('File input phải có cột customer_id.')

print(f'Input: {INPUT_PATH} ({len(raw_preview):,} rows)')
print(f'Output: {OUTPUT_PATH}')
raw_preview.head()

Input: E:\Personal Project\Data science_analyze finance\tests\data\sample.csv (50 rows)
Output: E:\Personal Project\Data science_analyze finance\data\predictions\sample_predictions.csv


,customer_id,age,income,credit_score,employment_years,employment_status,education_level,marital_status,home_ownership,region,...,num_credit_cards,num_loans,late_payment_months,debt_to_income,savings_rate,investment_value,account_creation_date,last_transaction_date,credit_risk,fraud_risk_score
0,FN0000001,74,65021.42,398,NaN,Retired,PhD,Single,Other,Southwest,...,0,0,20,0.547,0.505,205788.00,2019-06-10,2020-03-14,0,0.241
1,FN0000002,31,66975.00,572,35.8,Full-time,PhD,Divorced,Mortgage,Southwest,...,0,1,18,1.438,0.910,NaN,2022-10-25,2023-01-05,1,0.322
2,FN0000003,72,125906.64,999,5.6,Part-time,Bachelor,Widowed,Other,Southwest,...,2,3,13,1.493,0.587,42409.64,2022-03-24,2022-08-04,1,0.155
3,FN0000004,200,109125.14,481,22.2,Retired,High School,Widowed,Mortgage,Northeast,...,9,2,22,1.158,0.850,277730.86,2021-02-28,2021-08-18,3,0.874
4,FN0000005,72,145423.71,663,4.3,Self-employed,Bachelor,Married,Other,West,...,5,4,3,0.466,0.341,401029.89,2021-02-18,2021-08-19,0,0.283


## Chạy dự đoán

Hàm bên dưới nạp model bundle, áp dụng đúng feature engineering đã lưu cùng model và ghi file CSV theo cách an toàn.

In [3]:
predictions = run_batch_inference(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    task=TASK,
    version=MODEL_VERSION,
    model_root=PROJECT_ROOT / 'models',
)

print(f'Đã ghi {len(predictions):,} dự đoán vào: {OUTPUT_PATH}')
predictions.head()

{"timestamp": "2026-11-09T08:15:46.795281+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading CSV: E:\\Personal Project\\Data science_analyze finance\\tests\\data\\sample.csv"}
{"timestamp": "2026-11-09T08:15:46.872785+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid customer_id values to missing"}
{"timestamp": "2026-11-09T08:15:46.876495+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 1 invalid age values to missing"}
{"timestamp": "2026-11-09T08:15:46.878889+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 1 invalid income values to missing"}
{"timestamp": "2026-11-09T08:15:46.881644+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 1 invalid credit_score values to missing"}
{"timestamp": "2026-11-09T08:15:46.883898+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid employment_years values to missing"}
{"timestamp": "2026-11-09T08:15:46.885972+00:0

Đã ghi 50 dự đoán vào: E:\Personal Project\Data science_analyze finance\data\predictions\sample_predictions.csv


,customer_id,credit_risk_prediction,credit_risk_probability_0,credit_risk_probability_1,credit_risk_probability_2,credit_risk_probability_3,fraud_risk_score_prediction
0,FN0000001,3,0.001516,0.001435,0.050009,0.947041,0.536336
1,FN0000002,3,0.011422,0.024278,0.203421,0.760878,0.982468
2,FN0000003,3,0.013271,0.032834,0.155843,0.798052,0.965958
3,FN0000004,3,0.001078,0.005198,0.050746,0.942979,0.610290
4,FN0000005,2,0.312336,0.298837,0.327956,0.060871,0.440092


## Đọc kết quả

- `credit_risk_prediction`: lớp rủi ro tín dụng dự đoán (0 đến 3; giá trị càng cao càng rủi ro).
- `credit_risk_probability_*`: xác suất từng lớp credit risk.
- `fraud_risk_score_prediction`: điểm dự đoán fraud từ 0 đến 1; score càng cao càng nên ưu tiên rà soát.

In [4]:
display_columns = [column for column in [
    'customer_id',
    'credit_risk_prediction',
    'fraud_risk_score_prediction',
] if column in predictions.columns]

display(predictions[display_columns].head(10))

if 'credit_risk_prediction' in predictions.columns:
    display(predictions['credit_risk_prediction'].value_counts().sort_index().rename_axis('credit_risk').to_frame('customers'))

if 'fraud_risk_score_prediction' in predictions.columns:
    display(predictions['fraud_risk_score_prediction'].describe().to_frame('fraud_risk_score'))
    display(predictions.nlargest(10, 'fraud_risk_score_prediction')[display_columns])

,customer_id,credit_risk_prediction,fraud_risk_score_prediction
0,FN0000001,3,0.536336
1,FN0000002,3,0.982468
2,FN0000003,3,0.965958
3,FN0000004,3,0.610290
4,FN0000005,2,0.440092
5,FN0000006,3,1.183380
6,FN0000007,3,0.506481
7,FN0000008,3,1.104801
8,FN0000009,3,0.713332
9,FN0000010,3,0.990550


,customers
credit_risk,
0,7
1,4
2,5
3,34


,fraud_risk_score
count,50.000000
mean,0.715641
std,0.244732
min,0.299653
25%,0.527116
50%,0.679710
75%,0.958986
max,1.183380


,customer_id,credit_risk_prediction,fraud_risk_score_prediction
5,FN0000006,3,1.183380
30,FN0000031,3,1.150831
7,FN0000008,3,1.104801
17,FN0000018,3,1.084999
29,FN0000030,3,1.036812
40,FN0000041,0,0.998060
9,FN0000010,3,0.990550
13,FN0000014,1,0.990158
1,FN0000002,3,0.982468
38,FN0000039,3,0.980289


## Inference 1 khách hàng

In [11]:
from src.models.predict import load_model_bundle, predict_batch

# Lấy một dòng làm template để bảo đảm đủ schema input.
customer = pd.read_csv(PROJECT_ROOT / "tests/data/sample.csv").iloc[[40]].copy()

# # Thay thông tin khách hàng cần dự đoán tại đây.
# customer.loc[:, "age"] = 50
# customer.loc[:, "income"] = 38880.94
# # customer.loc[:, "credit_score"] = 680
# # customer.loc[:, "monthly_spending"] = 2_500
# # customer.loc[:, "credit_utilization"] = 0.45

credit_bundle = load_model_bundle(
    "v1.0.0", "credit_risk", PROJECT_ROOT / "models"
)
fraud_bundle = load_model_bundle(
    "v1.0.0", "fraud_risk_score", PROJECT_ROOT / "models"
)

credit_prediction = predict_batch(customer, credit_bundle)
fraud_prediction = predict_batch(customer, fraud_bundle)

single_prediction = credit_prediction.merge(
    fraud_prediction,
    on="customer_id",
    validate="one_to_one",
)
single_prediction

{"timestamp": "2026-11-09T08:40:18.999134+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid customer_id values to missing"}
{"timestamp": "2026-11-09T08:40:19.002202+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid age values to missing"}
{"timestamp": "2026-11-09T08:40:19.004556+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid income values to missing"}
{"timestamp": "2026-11-09T08:40:19.006996+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid credit_score values to missing"}
{"timestamp": "2026-11-09T08:40:19.009433+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid employment_years values to missing"}
{"timestamp": "2026-11-09T08:40:19.012302+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid account_age_months values to missing"}
{"timestamp": "2026-11-09T08:40:19.014569+00:00", "level": "INFO", "logger": "src.

,customer_id,credit_risk_prediction,credit_risk_probability_0,credit_risk_probability_1,credit_risk_probability_2,credit_risk_probability_3,fraud_risk_score_prediction
0,FN0000041,0,0.479049,0.071656,0.206523,0.242771,0.99806


## Inference one new customer

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

# Chạy độc lập, kể cả khi notebook đang mở trong thư mục notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.predict import load_model_bundle, predict_batch

# Lưu ý:
# - customer_id phải theo mẫu FN + 7 chữ số.
# - Không thêm credit_risk / fraud_risk_score vì đó là target cần dự đoán.
new_customer = {
    "customer_id": "FN1",
    "age": 35,
    "income": 120000.000,
    "credit_score": 680,
    "employment_years": 8,
    "employment_status": "Employed",
    "education_level": "Bachelor",
    "marital_status": "Married",
    "home_ownership": "Mortgage",
    "region": "Northeast",
    "account_age_months": 48,
    "monthly_spending": 2500.00,
    "credit_utilization": 0.45,
    "num_credit_cards": 3,
    "num_loans": 1,
    "late_payment_months": 1,
    "debt_to_income": 0.28,
    "savings_rate": 0.18,
    "investment_value": 25_000.00,
    "account_creation_date": "2022-11-01",
    "last_transaction_date": "2025-01-01",
}

customer = pd.DataFrame([new_customer])

# Kiểm tra schema theo dữ liệu train, không lấy giá trị từ khách hàng mẫu.
template = pd.read_csv(PROJECT_ROOT / "tests/data/sample.csv")
target_columns = {"credit_risk", "fraud_risk_score"}
expected_columns = set(template.columns) - target_columns

missing_columns = expected_columns - set(customer.columns)
extra_columns = set(customer.columns) - expected_columns
if missing_columns or extra_columns:
    raise ValueError(
        f"Thiếu cột: {sorted(missing_columns)} | "
        f"Cột không hợp lệ: {sorted(extra_columns)}"
    )

# Nạp hai model đã lưu
credit_bundle = load_model_bundle(
    version="v1.0.0",
    task="credit_risk",
    model_root=PROJECT_ROOT / "models",
)
fraud_bundle = load_model_bundle(
    version="v1.0.0",
    task="fraud_risk_score",
    model_root=PROJECT_ROOT / "models",
)

# Dự đoán
credit_prediction = predict_batch(customer, credit_bundle)
fraud_prediction = predict_batch(customer, fraud_bundle)

result = credit_prediction.merge(
    fraud_prediction,
    on="customer_id",
    validate="one_to_one",
)

display(result)

# Tùy chọn: lưu riêng kết quả một khách hàng
# output_path = PROJECT_ROOT / "data/predictions/new_customer_prediction.csv"
# output_path.parent.mkdir(parents=True, exist_ok=True)
# result.to_csv(output_path, index=False)

# print(f"Đã lưu kết quả: {output_path}")

{"timestamp": "2026-11-09T08:42:59.997412+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 1 invalid customer_id values to missing"}
{"timestamp": "2026-11-09T08:42:59.999507+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid age values to missing"}
{"timestamp": "2026-11-09T08:43:00.001173+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid income values to missing"}
{"timestamp": "2026-11-09T08:43:00.003383+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid credit_score values to missing"}
{"timestamp": "2026-11-09T08:43:00.006135+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid employment_years values to missing"}
{"timestamp": "2026-11-09T08:43:00.009404+00:00", "level": "INFO", "logger": "src.data.clean", "message": "Set 0 invalid account_age_months values to missing"}
{"timestamp": "2026-11-09T08:43:00.012290+00:00", "level": "INFO", "logger": "src.

,customer_id,credit_risk_prediction,credit_risk_probability_0,credit_risk_probability_1,credit_risk_probability_2,credit_risk_probability_3,fraud_risk_score_prediction
0,FN1,0,0.539995,0.342093,0.115878,0.002034,0.439396


Đã lưu kết quả: E:\Personal Project\Data science_analyze finance\data\predictions\new_customer_prediction.csv


## Chạy ngoài notebook

```powershell
.\.venv\Scripts\python.exe -m scripts.run_batch_inference `
  --input tests/data/sample.csv `
  --output data/predictions/sample_predictions.csv `
  --task both
```

Không dùng notebook này để train lại model; nó chỉ phục vụ inference từ model đã được duyệt.